In [1]:
"""
Predict on full PolNos corpus (1.44M sentences).
Filters to 6-59 word sentences before predicting (matches M&P).
Saves manifesto-level aggregated rates.
"""
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from tqdm import tqdm
import time

MODEL_DIR = "model_75"
CORPUS_PATH = "data_translated_full.parquet"   ## corpus and sentence-level output stay parquet: 1.19M rows exceed the xlsx row limit
OUTPUT_SENTENCES = "predictions_75_76_sentences_baseline.parquet"
OUTPUT_MANIFESTO = "../4. predict_data/predictions_75_76_manifesto_baseline.xlsx"
BATCH_SIZE = 512       
MAX_LEN = 84


In [2]:
## Load corpus
print("Loading corpus...")
corpus = pd.read_parquet(CORPUS_PATH)
print(f"  Total sentences: {len(corpus):,}")

## Drop missing text
corpus = corpus.dropna(subset=["text"]).reset_index(drop=True)
print(f"  After dropping null text: {len(corpus):,}")

## Filter to 6-59 word sentences (M&P's filter)
corpus["ntoken"] = corpus["text"].str.split().str.len()
mask = (corpus["ntoken"] >= 6) & (corpus["ntoken"] <= 59)
corpus = corpus[mask].reset_index(drop=True)
print(f"  After 6-59 word filter: {len(corpus):,}")


Loading corpus...
  Total sentences: 1,439,551
  After dropping null text: 1,439,551
  After 6-59 word filter: 1,195,743


In [3]:
## Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_DIR)
model = RobertaForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()

## Optional: half precision for speed (a Pro 6000 supports it well)
USE_FP16 = True
if USE_FP16:
    model = model.half()
    print("Using FP16 inference")

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }

texts = corpus["text"].tolist()
ds = TextDataset(texts, tokenizer, MAX_LEN)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


Device: cuda


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Using FP16 inference


In [4]:
## Predict
all_preds = []
all_probas = []
t0 = time.time()
with torch.no_grad():
    for batch in tqdm(loader, desc="Predicting"):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits.float()
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(logits, dim=-1)
        all_preds.append(preds.cpu().numpy())
        all_probas.append(probs[:, 1].cpu().numpy())

elapsed = time.time() - t0
print(f"\nTotal inference time: {elapsed/60:.1f} min")


Predicting: 100%|██████████| 2336/2336 [02:35<00:00, 14.98it/s]


Total inference time: 2.6 min


In [5]:

corpus["pred_label"] = np.concatenate(all_preds).astype(int)
corpus["pred_proba"] = np.concatenate(all_probas).astype(float)

## Save sentence-level (small subset of columns to keep file size down)
out_cols = ["manifesto_id", "party", "edate", "countryname", "doc_id", "pos",
            "ntoken", "cmp_code", "pred_label", "pred_proba"]
corpus[out_cols].to_parquet(OUTPUT_SENTENCES, index=False)
print(f"Saved sentence-level predictions to {OUTPUT_SENTENCES}")

## Aggregate to manifesto level
manifesto_agg = corpus.groupby("manifesto_id").agg(
    party=("party", "first"),
    edate=("edate", "first"),
    countryname=("countryname", "first"),
    n_sentences_filtered=("pred_label", "count"),
    n_nostalgic_pred=("pred_label", "sum"),
).reset_index()
manifesto_agg["nostalgic_per_1000_50_101"] = (
    manifesto_agg["n_nostalgic_pred"] / manifesto_agg["n_sentences_filtered"] * 1000
)
manifesto_agg.to_excel(OUTPUT_MANIFESTO, index=False)
print(f"Saved manifesto-level rates to {OUTPUT_MANIFESTO}")

print("\nManifesto-level rate summary:")
print(manifesto_agg["nostalgic_per_1000_50_101"].describe())


Saved sentence-level predictions to predictions_75_76_sentences_baseline.parquet
Saved manifesto-level rates to predictions_75_76_manifesto_baseline.parquet

Manifesto-level rate summary:
count    1648.000000
mean       25.430394
std        28.778812
min         0.000000
25%         9.226081
50%        19.083969
75%        32.677196
max       318.181818
Name: nostalgic_per_1000_50_101, dtype: float64
